# Purpose 

The purpose of this notebook is to ingest raw data, perform basic data quality checks, and apply light ETL processes. The goal is to prepare the raw data for the bronze layer, enabling more detailed ETL operations at a later stage.

**In this notebook, we will perform the following tasks:**

- Read the raw data
- Conduct basic data quality checks
- Apply basic ETL processes
- Save the final DataFrame as a CSV file

# Import

Import necessary packages

In [42]:
from src.utils import read_sheets_from_excel_file, describe_df, save_df_as_csv, convert_excel_serial_date_to_datetime

Let's begin by reviewing the available data.
We have an Excel file that contains multiple sheets.

In [43]:

# Load .xlsb file
file_path = "DS_task_data.xlsb"
dataset_name = f"dataset/raw/{file_path}"
sheet_names=['customers', 'bills']

We'll start by reading each sheet into its own DataFrame. After that, we will examine each DataFrame individually.

In [44]:

# Load customers sheet
customers_df = read_sheets_from_excel_file(dataset_name,sheet_names[0] )

# Load bills sheet
bills_df = read_sheets_from_excel_file(dataset_name,sheet_names[1] )


## Customers Dataset

Let's begin by reviewing the customer information. 

We will utilise the functions defined in the utils module to review the data status. This includes viewing sample records, data types, missing values, and the unique values for each column.

In [45]:

describe_df(customers_df,"Customers" )


 --- Customers ---:


,customer_account,sme_mbs,segment_id,cust_group,payment_method,internal_credit_rating,experian_credit_score,sales_route,segment,count_sites,tenure_group,contracted_annual_volume
0,880000057759,MBS,WHOLESALE,LTD,DD,Great payer,40.0,Direct,VBR,2,1-3 years,94713
1,880000079664,SME,MANUFACTURING,LTD,DD,Great payer,62.0,Indirect,Fixed,1,<90 days,1613308
2,880000058066,SME,BEAUTY,LTD,DD,Great payer,43.0,Direct,Fixed,1,6 months - 1 year,5370
3,880000014133,SME,PROFESSIONAL_SPECIAL,LTD,DD,Great payer,85.0,Direct,Fixed,1,1-3 years,4115
4,880000079447,MBS,PROPERTY,LTD,NONDD,Late payer,0.0,Direct,VBR,1,1-3 years,3098


customer_account              int64
sme_mbs                      object
segment_id                   object
cust_group                   object
payment_method               object
internal_credit_rating       object
experian_credit_score       float64
sales_route                  object
segment                      object
count_sites                   int64
tenure_group                 object
contracted_annual_volume      int64
dtype: object


 -- Missing Values - Customers --- 


customer_account               0
sme_mbs                        0
segment_id                     1
cust_group                     0
payment_method                 0
internal_credit_rating         0
experian_credit_score       3376
sales_route                    0
segment                        0
count_sites                    0
tenure_group                   0
contracted_annual_volume       0
dtype: int64


 -- Unique value counts --- 
customer_account: 10000 unique values
sme_mbs: 2 unique values
segment_id: 66 unique values
cust_group: 18 unique values
payment_method: 5 unique values
internal_credit_rating: 6 unique values
experian_credit_score: 101 unique values
sales_route: 2 unique values
segment: 3 unique values
count_sites: 80 unique values
tenure_group: 6 unique values
contracted_annual_volume: 8034 unique values


As observed above, there are some missing values, particularly in the experian_credit_score column and one missing segment_id. However, the rest of the data appears to be in good condition.

## Bills dataset

Let's repeat the save process for the bills DataFrame as well.

In [46]:
describe_df(bills_df,"Bills" )


 --- Bills ---:


,customer_account,invoice_voucher,invoice_date,due_date,amount,amount_settled,settled_date,balance
0,880000000010,IV00000097,45048,45062,52.26,52.26,45062.0,0.0
1,880000000010,IV00006738,45131,45145,48.25,48.25,45145.0,0.0
2,880000000010,IV00047259,45230,45244,57.26,57.26,45261.0,0.0
3,880000000010,IV00314258,45337,45351,1565.57,1565.57,45351.0,0.0
4,880000000010,IV00801038,45410,45424,1295.81,1295.81,45425.0,0.0


customer_account      int64
invoice_voucher      object
invoice_date          int64
due_date              int64
amount              float64
amount_settled      float64
settled_date        float64
balance             float64
dtype: object


 -- Missing Values - Bills --- 


customer_account        0
invoice_voucher         0
invoice_date            0
due_date                0
amount                  0
amount_settled          0
settled_date        20368
balance                 0
dtype: int64


 -- Unique value counts --- 
customer_account: 9999 unique values
invoice_voucher: 177220 unique values
invoice_date: 559 unique values
due_date: 639 unique values
amount: 95755 unique values
amount_settled: 87802 unique values
settled_date: 601 unique values
balance: 19430 unique values


Similar to the previous dataset, most columns appear to be in good condition. However, there are missing values in the settled_date column. Additionally, the invoice_date, due_date, and settled_date fields contain Excel serial date formats, which need to be converted to human-readable dates.

In [47]:

# Convert excel serial dates 
date_cols = ['invoice_date', 'due_date', 'settled_date']
for col in date_cols:
    bills_df[col] = bills_df[col].apply(convert_excel_serial_date_to_datetime)


In [48]:
bills_df

,customer_account,invoice_voucher,invoice_date,due_date,amount,amount_settled,settled_date,balance
0,880000000010,IV00000097,2023-05-02,2023-05-16,52.26,52.26,2023-05-16,0.00
1,880000000010,IV00006738,2023-07-24,2023-08-07,48.25,48.25,2023-08-07,0.00
2,880000000010,IV00047259,2023-10-31,2023-11-14,57.26,57.26,2023-12-01,0.00
3,880000000010,IV00314258,2024-02-15,2024-02-29,1565.57,1565.57,2024-02-29,0.00
4,880000000010,IV00801038,2024-04-28,2024-05-12,1295.81,1295.81,2024-05-13,0.00
...,...,...,...,...,...,...,...,...
177215,880000240410,IV02651398,2025-03-19,2025-04-02,5604.36,0.00,NaT,5604.36
177216,880000240410,IV02665601,2025-03-21,2025-04-04,719.04,0.00,NaT,719.04
177217,880000243959,IV02684842,2025-03-28,2025-04-11,100.16,0.00,NaT,100.16
177218,880000243959,IV02687134,2025-03-29,2025-04-12,144.48,0.00,NaT,144.48


As shown above, the date conversion was successful. However, there is still an issue with missing values in the settled_date column.

# Save the DataFrame

Let's save this DataFrame to the bronze folder. It will serve as the input data for the next stage of the ETL process.

In [49]:

save_df_as_csv(customers_df, "dataset/bronze/bronze_customers.csv")
save_df_as_csv(bills_df, "dataset/bronze/bronze_bills.csv")